# Family Relationships Ontology

This example is a part of [AI for Beginners Curriculum](http://github.com/microsoft/ai-for-beginners), and it has been inspired by [this blog post](https://habr.com/post/270857/).

I always find it difficult to remember different relationships between people in a family. In this example, we will take an ontology that defines family relationships, and the actual genealogical tree, and show how we can then perform automatic inference to find all relatives.

### Getting the Genealogical Tree

As an example, we will take genealogical tree of [Romanov Tsar Family](https://en.wikipedia.org/wiki/House_of_Romanov). The most common format for describing family relationships is [GEDCOM](https://en.wikipedia.org/wiki/GEDCOM). We will take Romanov family tree in GEDCOM format:

In [1]:
!head -15 data/tsars.ged

'head' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


To use GEDCOM file, we can use `python-gedcom` library:

In [2]:
import sys
!{sys.executable} -m pip install python-gedcom

This library takes away some of the technical problems with file parsing, but it still gives us pretty low-level access to all individuals and families in the tree. Here is how we can parse the file, and show the list of all individuals:

In [5]:
from gedcom.parser import Parser
from gedcom.element.individual import IndividualElement
from gedcom.element.family import FamilyElement
g = Parser()
g.parse_file('data/tsars.ged')

In [3]:
d = g.get_element_dictionary()
[ (k,v.get_name()) for k,v in d.items() if isinstance(v,IndividualElement)]

[('@0@', ('Mihail Fedorovich', 'Romanov')),
 ('@1@', ('Evdokija Lukjanovna', 'Streshneva')),
 ('@2@', ('Aleksej Mihajlovich', 'Romanov')),
 ('@3@', ('Marija Ilinichna', 'Miloslavskaja')),
 ('@4@', ('Natalja Kirillovna', 'Naryshkina')),
 ('@5@', ('Marfa Matveevna', 'Apraksina')),
 ('@6@', ('Fedor Alekseevich', 'Romanov')),
 ('@7@', ('Sofja Aleksevna', 'Romanova')),
 ('@8@', ('Ivan V Alekseevich', 'Romanov')),
 ('@9@', ('Praskovja Fedorovna', 'Saltykova')),
 ('@10@', ('Ekaterina Ivanovna', 'Romanova')),
 ('@11@', ('Anna Ivanovna', 'Romanova')),
 ('@12@', ('Fridrih Vilgelm', 'Kurlandskij')),
 ('@13@', ('Karl Leopold', 'Meklenburg-Shverinskij')),
 ('@14@', ('Anna Leopoldovna', 'Meklenburg-Shverinskaja')),
 ('@15@', ('Anton Ulrih', 'Braunshvejg-Volfenbjuttelskij')),
 ('@16@', ('Ivan VI Antonovich', 'Braunshvejg-Volfenbjuttelskij')),
 ('@17@', ('Petr I Alekseevich', 'Romanov')),
 ('@18@', ('Evdokija Fedorovna', 'Lopuhina')),
 ('@19@', ('Ekaterina I Alekseevna', 'Mihajlova')),
 ('@20@', ('Ale

Here is how we can get information about families. Note that is gives us a list of **identifiers**, and we need to convert them to names if we want more clarity:

In [4]:
d = g.get_element_dictionary()
[ (k,[x.get_value() for x in v.get_child_elements()]) for k,v in d.items() if isinstance(v,FamilyElement)]

[('@41@', ['@0@', '@1@', '@2@']),
 ('@42@', ['@2@', '@3@', '@6@', '@7@', '@8@']),
 ('@43@', ['@8@', '@9@', '@10@', '@11@']),
 ('@44@', ['@13@', '@10@', '@14@']),
 ('@45@', ['@15@', '@14@', '@16@']),
 ('@46@', ['@2@', '@4@', '@17@']),
 ('@47@', ['@17@', '@18@', '@20@']),
 ('@48@', ['@20@', '@21@', '@22@']),
 ('@49@', ['@17@', '@19@', '@23@', '@24@']),
 ('@50@', ['@25@', '@23@', '@26@']),
 ('@51@', ['@26@', '@27@', '@28@']),
 ('@52@', ['@28@', '@30@', '@31@', '@33@']),
 ('@53@', ['@33@', '@34@', '@35@']),
 ('@54@', ['@35@', '@36@', '@37@']),
 ('@55@', ['@37@', '@38@', '@39@'])]

### Getting Family Ontology

Next, let's have a look at [family ontology](https://raw.githubusercontent.com/blokhin/genealogical-trees/master/data/header.ttl) defined as a set of Semantic Web triplets. This ontology defines such relationships as `isUncleOf`, `isCousinOf`, and many others. All those relationships are defined in terms of basic predicates `isMotherOf`, `isFatherOf`, `isBrotherOf` and `isSisterOf`. We will use automatic reasoning to deduce all other relationships using the ontology.

Here is a sample definition of `isAuntOf` property, which is defined as a composition of `isSisterOf` and `isParentOf` (*Aunt is a sister of one's parent*).

```
fhkb:isAuntOf a owl:ObjectProperty ;
    rdfs:domain fhkb:Woman ;
    rdfs:range fhkb:Person ;
    owl:propertyChainAxiom ( fhkb:isSisterOf fhkb:isParentOf ) .
```

In [6]:
!head -20 data/onto.ttl

'head' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


### Constructing Ontology for Inference

For simplicity, we will create one ontology file that will include original rules from family ontology, and facts about individuals from our GEDCOM file. We will go through the GEDCOM file and extract information about families and individuals, and convert them to triplets.

In [7]:
# 这行代码看起来像是在执行系统命令，将data目录下的onto.ttl文件复制到当前目录
# 假设是在支持!执行系统命令的环境，如Jupyter Notebook
!cp data/onto.ttl.

# 获取解析GEDCOM文件后的元素字典
gedcom_dict = g.get_element_dictionary()

# 初始化两个空字典，分别用于存储个体信息和婚姻信息
individuals, marriages = {}, {}

# 定义一个函数，将GEDCOM元素的指针转换为特定格式的标识符
# 用于在TTL文件中作为个体或家庭的唯一标识
def term2id(el):
    return "i" + el.get_pointer().replace('@', '').lower()

# 以追加模式打开onto.ttl文件，准备写入转换后的本体数据
out = open("onto.ttl", "a")

# 遍历解析后的GEDCOM元素字典
for k, v in gedcom_dict.items():
    # 如果当前元素是IndividualElement（个体元素）
    if isinstance(v, IndividualElement):
        # 初始化两个集合，分别用于存储该个体的子女和兄弟姐妹的标识符
        children, siblings = set(), set()
        # 获取个体的唯一标识符
        idx = term2id(v)

        # 从个体的姓名信息生成title，并对其进行一些字符串清理操作
        title = v.get_name()[0] + " " + v.get_name()[1]
        title = title.replace('"', '').replace('[', '').replace(']', '').replace('(', '').replace(')', '').strip()

        # 获取该个体作为配偶的家庭列表
        own_families = g.get_families(v, 'FAMS')
        # 遍历这些家庭，获取子女的标识符并添加到children集合
        for fam in own_families:
            children |= set(term2id(i) for i in g.get_family_members(fam, "CHIL"))

        # 获取该个体作为子女的家庭列表
        parent_families = g.get_families(v, 'FAMC')
        # 如果存在这样的家庭
        if len(parent_families):
            # 遍历家庭成员获取兄弟姐妹的标识符并添加到siblings集合
            # 这里未处理收养家庭（即家庭数量大于1时未做特殊处理，代码中有TODO注释）
            for member in g.get_family_members(parent_families[0], "CHIL"):
                if member.get_pointer() == v.get_pointer():
                    continue
                siblings.add(term2id(member))

        # 如果idx已经存在于individuals字典中，合并新获取的子女和兄弟姐妹集合
        if idx in individuals:
            children |= individuals[idx].get('children', set())
            siblings |= individuals[idx].get('siblings', set())
        # 将个体的性别、子女、兄弟姐妹和标题信息存储到individuals字典中
        individuals[idx] = {'sex': v.get_gender().lower(), 'children': children,'siblings': siblings, 'title': title}

    # 如果当前元素是FamilyElement（家庭元素）
    elif isinstance(v, FamilyElement):
        # 初始化妻子、丈夫和子女的变量
        wife, husb, children = None, None, set()
        # 获取家庭中的子女标识符并添加到children集合
        children = set(term2id(i) for i in g.get_family_members(v, "CHIL"))

        # 尝试获取家庭中的妻子信息
        try:
            wife = g.get_family_members(v, "WIFE")[0]
            wife = term2id(wife)
            # 如果妻子的标识符已经存在于individuals字典中，合并子女集合
            if wife in individuals:
                individuals[wife]['children'] |= children
            else:
                individuals[wife] = {'children': children}
        # 如果获取妻子信息时发生索引错误（即没有妻子），跳过
        except IndexError:
            pass

        # 尝试获取家庭中的丈夫信息
        try:
            husb = g.get_family_members(v, "HUSB")[0]
            husb = term2id(husb)
            # 如果丈夫的标识符已经存在于individuals字典中，合并子女集合
            if husb in individuals:
                individuals[husb]['children'] |= children
            else:
                individuals[husb] = {'children': children}
        # 如果获取丈夫信息时发生索引错误（即没有丈夫），跳过
        except IndexError:
            pass

        # 如果妻子和丈夫都存在，记录婚姻信息
        if wife and husb:
            marriages[wife + husb] = (term2id(v), wife, husb)

# 遍历个体信息字典
for idx, val in individuals.items():
    # 初始化一个字符串，用于存储添加到TTL文件中的额外关系描述
    added_terms = ''
    # 根据个体性别确定父母关系和兄弟姐妹关系的谓词
    if val['sex'] == 'f':
        parent_predicate, sibl_predicate = "isMotherOf", "isSisterOf"
    else:
        parent_predicate, sibl_predicate = "isFatherOf", "isBrotherOf"

    # 如果个体有子女，添加子女关系描述到added_terms
    if len(val['children']):
        added_terms += " ;\n    fhkb:" + parent_predicate + " " + ", ".join(["fhkb:" + i for i in val['children']])
    # 如果个体有兄弟姐妹，添加兄弟姐妹关系描述到added_terms
    if len(val['siblings']):
        added_terms += " ;\n    fhkb:" + sibl_predicate + " " + ", ".join(["fhkb:" + i for i in val['siblings']])
    # 将个体的信息写入onto.ttl文件
    out.write("fhkb:%s a owl:NamedIndividual, owl:Thing%s ;\n    rdfs:label \"%s\" .\n" % (idx, added_terms, val['title']))

# 遍历婚姻信息字典
for k, v in marriages.items():
    # 将婚姻信息写入onto.ttl文件
    out.write("fhkb:%s a owl:NamedIndividual, owl:Thing ;\n    fhkb:hasFemalePartner fhkb:%s ;\n    fhkb:hasMalePartner fhkb:%s .\n" % v)

# 写入OWL的AllDifferent声明，确保所有个体和婚姻的标识符是不同的
out.write("[] a owl:AllDifferent ;\n    owl:distinctMembers (")
# 写入所有个体的标识符
for idx in individuals.keys():
    out.write("    fhkb:" + idx)
# 写入所有婚姻的标识符
for k, v in marriages.items():
    out.write("    fhkb:" + v[0])
# 结束OWL的AllDifferent声明
out.write("    ) .")
# 关闭文件
out.close()  

In [8]:
!tail onto.ttl

    fhkb:hasFemalePartner fhkb:i34 ;
    fhkb:hasMalePartner fhkb:i33 .
fhkb:i54 a owl:NamedIndividual, owl:Thing ;
    fhkb:hasFemalePartner fhkb:i36 ;
    fhkb:hasMalePartner fhkb:i35 .
fhkb:i55 a owl:NamedIndividual, owl:Thing ;
    fhkb:hasFemalePartner fhkb:i38 ;
    fhkb:hasMalePartner fhkb:i37 .
[] a owl:AllDifferent ;
    owl:distinctMembers (    fhkb:i0    fhkb:i1    fhkb:i2    fhkb:i3    fhkb:i4    fhkb:i5    fhkb:i6    fhkb:i7    fhkb:i8    fhkb:i9    fhkb:i10    fhkb:i11    fhkb:i12    fhkb:i13    fhkb:i14    fhkb:i15    fhkb:i16    fhkb:i17    fhkb:i18    fhkb:i19    fhkb:i20    fhkb:i21    fhkb:i22    fhkb:i23    fhkb:i24    fhkb:i25    fhkb:i26    fhkb:i27    fhkb:i28    fhkb:i29    fhkb:i30    fhkb:i31    fhkb:i32    fhkb:i33    fhkb:i34    fhkb:i35    fhkb:i36    fhkb:i37    fhkb:i38    fhkb:i39    fhkb:i40    fhkb:i41    fhkb:i42    fhkb:i43    fhkb:i44    fhkb:i45    fhkb:i46    fhkb:i47    fhkb:i48    fhkb:i49    fhkb:i50    fhkb:i51    fhkb:i52    fhkb:i53    fhkb:

### Doing Inference 

Now we want to be able to use this ontology for inference and for querying. We will use [RDFLib](https://github.com/RDFLib), library for reading RDF Graph in different formats, querying it, etc. 

For logical inference, we will use [OWL-RL](https://github.com/RDFLib/OWL-RL) library, which allows us to build **Closure** of the RDF Graph, i.e. add all possible concepts and relations that can be inferred.

In [10]:
!{sys.executable} -m pip install rdflib
!{sys.executable} -m pip install git+https://github.com/RDFLib/OWL-RL.git

  Cloning https://github.com/RDFLib/OWL-RL.git to /tmp/pip-req-build-lbfzwi3m
  Running command git clone --filter=blob:none --quiet https://github.com/RDFLib/OWL-RL.git /tmp/pip-req-build-lbfzwi3m
  Resolved https://github.com/RDFLib/OWL-RL.git to commit a77e1791b88b54aace609bc6000aac14c7add4ff
  Preparing metadata (setup.py) ... done


Let's open the ontology file and see how many triplets it contains:

In [11]:
import rdflib
from owlrl import DeductiveClosure, OWLRL_Extension

g = rdflib.Graph()
g.parse("onto.ttl", format="turtle")

print("Triplets found:%d" % len(g))

Triplets found:669


Now let's build the closure, and see how the number of triplets increase:

In [12]:
DeductiveClosure(OWLRL_Extension).expand(g)
print("Triplets after inference:%d" % len(g))

Triplets after inference:4246


### Querying for Relatives 

Now we can query the graph to see different relations between people. We can use **SPARQL** language together with `query` method. In our case, let's see all **uncles** in our family tree:

In [13]:
qres = g.query(
    """SELECT DISTINCT ?aname ?bname
       WHERE {
          ?a fhkb:isUncleOf ?b .
          ?a rdfs:label ?aname .
          ?b rdfs:label ?bname .
       }""")

for row in qres:
    print("%s is uncle of %s" % row)

Fedor Alekseevich Romanov is uncle of Ekaterina Ivanovna Romanova
Aleksandr I Pavlovich Romanov is uncle of Aleksandr II Nikolaevich Romanov
Fedor Alekseevich Romanov is uncle of Anna Ivanovna Romanova


Feel free to experiment with different other family relations. For example, you can have a look at `isAncestorOf` relation, which recurrently defines all ancestors of a given person.

Finally, let's clean up!

In [14]:
!rm onto.ttl